# CKPT6: Alternative Techniques vs CKPT2 Baselines


We evaluate **four alternative techniques** on the same temporal splits and compare to CKPT2 baselines:

1. Two-stage (Hurdle) model
2. Count-aware objectives (Poisson / Tweedie / XGBoost / LightGBM if available)
3. Probabilistic CLV (BG/NBD + Gamma-Gamma)
4. Sequence model (monthly counts -> MLP proxy)


In [1]:
# Setup
import sys
sys.path.append('..')


In [2]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

from src.data import load_online_retail_ii, clean_data
from src.features import create_temporal_splits_multi
from src.baselines import (
    evaluate_model,
    train_bgnbd_baseline,
    predict_bgnbd,
)
from src.two_stage import TwoStageModel

from sklearn.ensemble import HistGradientBoostingRegressor


## Data + Temporal Splits (same as CKPT2)


In [3]:
file_2009 = Path('../data/Year 2009-2010.csv')
file_2010 = Path('../data/Year 2010-2011.csv')

if not (file_2009.exists() and file_2010.exists()):
    raise FileNotFoundError('Raw dataset files missing in ../data/')

raw = load_online_retail_ii(file_2009, file_2010)
clean = clean_data(raw)

train_cutoffs = ['2010-06-01','2010-09-01','2010-12-01','2011-03-01']
val_cutoff = '2011-06-01'
test_cutoff = '2011-09-01'

obs_months = 6
horizon_months = 3

train_df, val_df, test_df = create_temporal_splits_multi(
    clean, train_cutoffs, val_cutoff, test_cutoff, obs_months=obs_months, horizon_months=horizon_months
)

exclude = {'customer_id','CustomerID','target','cutoff_date','horizon_start','horizon_end'}
feature_cols = [c for c in train_df.columns if c not in exclude]

X_train = train_df[feature_cols]
y_train = train_df['target']
X_val = val_df[feature_cols]
y_val = val_df['target']
X_test = test_df[feature_cols]
y_test = test_df['target']


Loading Year 2009-2010...


  Loaded 525,461 rows
Loading Year 2010-2011...


  Loaded 541,910 rows

Total rows: 1,067,371

DATA CLEANING PIPELINE
Initial rows: 1,067,371
✓ Removed missing CustomerID: 243,007 rows removed


✓ Removed cancellations: 18,744 rows removed
✓ Removed invalid Quantity/Price: 71 rows removed


✓ Converted InvoiceDate to datetime


...

  Final customers with features: 2,868
  Customers with target > 0: 1,779 (62.0%)

--- Train cutoff: 2010-12-01 ---

Creating window:
  Observation: 2010-06-01 to 2010-12-01 (6 months)
  Horizon: 2010-12-01 to 2011-03-01 (3 months)
  Observation transactions: 225,116
  Horizon transactions: 66,364
  Customers in observation: 3,497


  Final customers with features: 3,497
  Customers with target > 0: 1,324 (37.9%)

--- Train cutoff: 2011-03-01 ---

Creating window:
  Observation: 2010-09-01 to 2011-03-01 (6 months)
  Horizon: 2011-03-01 to 2011-06-01 (3 months)
  Observation transactions: 208,120
  Horizon transactions: 77,376
  Customers in observation: 3,358


  Final customers with features: 3,358
  Customers with target > 0: 1,425 (42.4%)

[2/3] Validation Set

Creating window:
  Observation: 2010-12-01 to 2011-06-01 (6 months)
  Horizon: 2011-06-01 to 2011-09-01 (3 months)
  Observation transactions: 143,740
  Horizon transactions: 80,296
  Customers in observation: 2,718


  Final customers with features: 2,718
  Customers with target > 0: 1,349 (49.6%)

[3/3] Test Set

Creating window:
  Observation: 2011-03-01 to 2011-09-01 (6 months)
  Horizon: 2011-09-01 to 2011-12-01 (3 months)
  Observation transactions: 157,672
  Horizon transactions: 151,630
  Customers in observation: 2,813


  Final customers with features: 2,813
  Customers with target > 0: 1,701 (60.5%)

SPLIT SUMMARY
Train (multi-cutoff): 12,426 rows
Val:                  2,718 rows
Test:                 2,813 rows



## Baseline Reference (from CKPT2 artifacts if available)


In [4]:
baseline_ref = None
ckpt2_path = Path('../results/ckpt2/baseline_metrics.json')
if ckpt2_path.exists():
    with open(ckpt2_path, 'r') as f:
        baseline_ref = json.load(f)

if baseline_ref:
    baseline_rows = []
    for k, v in baseline_ref.items():
        baseline_rows.append({'Model': k, 'MAE': v['MAE'], 'RMSE': v['RMSE']})
    baseline_df = pd.DataFrame(baseline_rows).sort_values('MAE')
    print('Loaded CKPT2 baselines from artifacts.')
else:
    baseline_df = pd.DataFrame([])
    print('No CKPT2 baseline artifact found. Will compare against new techniques only.')

baseline_df


Loaded CKPT2 baselines from artifacts.


,Model,MAE,RMSE
1,results_rf_test,1.106082,1.958634
3,results_avg_test,1.110828,1.958989
9,results_mlp_test,1.115126,1.801244
2,results_xgb_test,1.118455,1.850803
0,results_en_test,1.122049,2.160581
5,results_hgb_test,1.130360,2.249780
4,results_et_test,1.180362,2.058104
8,results_svr_test,1.208783,2.641880
6,results_pr_test,1.221881,1.920366
7,results_knn_test,1.261459,2.619688


## Technique 1: Two?Stage (Hurdle) Model


In [5]:
two_stage = TwoStageModel()
two_stage.fit(X_train, y_train)

pred_two_val = two_stage.predict(X_val)
pred_two_test = two_stage.predict(X_test)

res_two_val = evaluate_model(y_val, pred_two_val, 'TwoStage')
res_two_test = evaluate_model(y_test, pred_two_test, 'TwoStage')

print('Val:', res_two_val)
print('Test:', res_two_test)


Val: {'Model': 'TwoStage', 'MAE': 0.8966952305414377, 'RMSE': np.float64(1.4110822299337773)}
Test: {'Model': 'TwoStage', 'MAE': 1.111556039038797, 'RMSE': np.float64(1.9893552040761313)}


## Technique 2: Count-Aware Objectives (Poisson / Tweedie / XGBoost / LightGBM)

In [6]:
results_count = {}

from sklearn.linear_model import TweedieRegressor

# HistGradientBoosting with Poisson loss
poisson_hgb = HistGradientBoostingRegressor(loss='poisson', max_depth=6, learning_rate=0.1, random_state=42)
poisson_hgb.fit(X_train, y_train)

pred_hgb_val = poisson_hgb.predict(X_val)
pred_hgb_test = poisson_hgb.predict(X_test)
results_count['Poisson_HGB_val'] = evaluate_model(y_val, pred_hgb_val, 'Poisson_HGB')
results_count['Poisson_HGB_test'] = evaluate_model(y_test, pred_hgb_test, 'Poisson_HGB')

# Tweedie (approx Negative Binomial / compound Poisson)
try:
    tweedie = TweedieRegressor(power=1.5, alpha=0.1, link='log', max_iter=1000)
    tweedie.fit(X_train, y_train)
    pred_tw_val = tweedie.predict(X_val)
    pred_tw_test = tweedie.predict(X_test)
    results_count['Tweedie_val'] = evaluate_model(y_val, pred_tw_val, 'Tweedie')
    results_count['Tweedie_test'] = evaluate_model(y_test, pred_tw_test, 'Tweedie')
except Exception as e:
    print(f'Tweedie failed: {e}')

# Optional: XGBoost count:poisson
try:
    import xgboost as xgb
    xgb_count = xgb.XGBRegressor(objective='count:poisson', n_estimators=300, max_depth=5, learning_rate=0.1, random_state=42, n_jobs=-1)
    xgb_count.fit(X_train, y_train)
    pred_xgb_val = xgb_count.predict(X_val)
    pred_xgb_test = xgb_count.predict(X_test)
    results_count['XGB_Poisson_val'] = evaluate_model(y_val, pred_xgb_val, 'XGB_Poisson')
    results_count['XGB_Poisson_test'] = evaluate_model(y_test, pred_xgb_test, 'XGB_Poisson')
except Exception:
    print('XGBoost not available for count:poisson')

# Optional: LightGBM count objectives
try:
    import lightgbm as lgb
    lgb_poisson = lgb.LGBMRegressor(objective='poisson', n_estimators=300, learning_rate=0.05, num_leaves=31, random_state=42)
    lgb_poisson.fit(X_train, y_train)
    pred_lgb_val = lgb_poisson.predict(X_val)
    pred_lgb_test = lgb_poisson.predict(X_test)
    results_count['LGB_Poisson_val'] = evaluate_model(y_val, pred_lgb_val, 'LGB_Poisson')
    results_count['LGB_Poisson_test'] = evaluate_model(y_test, pred_lgb_test, 'LGB_Poisson')

    lgb_tweedie = lgb.LGBMRegressor(objective='tweedie', tweedie_variance_power=1.5, n_estimators=300, learning_rate=0.05, num_leaves=31, random_state=42)
    lgb_tweedie.fit(X_train, y_train)
    pred_lgb_tw_val = lgb_tweedie.predict(X_val)
    pred_lgb_tw_test = lgb_tweedie.predict(X_test)
    results_count['LGB_Tweedie_val'] = evaluate_model(y_val, pred_lgb_tw_val, 'LGB_Tweedie')
    results_count['LGB_Tweedie_test'] = evaluate_model(y_test, pred_lgb_tw_test, 'LGB_Tweedie')
except Exception:
    print('LightGBM not available (optional)')

for k, v in results_count.items():
    print(k, v)


Poisson_HGB_val {'Model': 'Poisson_HGB', 'MAE': 0.9022379631353122, 'RMSE': np.float64(1.4142786371957143)}
Poisson_HGB_test {'Model': 'Poisson_HGB', 'MAE': 1.1295246611329033, 'RMSE': np.float64(2.271224033059876)}
XGB_Poisson_val {'Model': 'XGB_Poisson', 'MAE': 0.9034039378166199, 'RMSE': np.float64(1.4632193956066757)}
XGB_Poisson_test {'Model': 'XGB_Poisson', 'MAE': 1.1385142803192139, 'RMSE': np.float64(2.0875709795734085)}


## Technique 3: Probabilistic CLV (BG/NBD + Gamma-Gamma)

In [7]:
res_bgnbd_val = res_bgnbd_test = None

# BG/NBD for counts (same target as baselines)
model_bgnbd = train_bgnbd_baseline(train_df, horizon_months=3)
if model_bgnbd is not None:
    pred_bgnbd_val = predict_bgnbd(model_bgnbd, val_df, horizon_months=3)
    pred_bgnbd_test = predict_bgnbd(model_bgnbd, test_df, horizon_months=3)
    res_bgnbd_val = evaluate_model(y_val, pred_bgnbd_val, 'BG/NBD')
    res_bgnbd_test = evaluate_model(y_test, pred_bgnbd_test, 'BG/NBD')
    print('BG/NBD (count) test:', res_bgnbd_test)

# Gamma-Gamma for monetary CLV (separate metric)
res_gg_test = None
try:
    from lifetimes import BetaGeoFitter, GammaGammaFitter
    from lifetimes.utils import summary_data_from_transaction_data

    # Build monetary value for transactions
    df_money = clean.copy()
    if 'Sales' not in df_money.columns:
        df_money['Sales'] = df_money['Quantity'] * df_money['Price']

    cutoff = pd.to_datetime(test_cutoff)
    obs_start = cutoff - pd.DateOffset(months=obs_months)
    horizon_end = cutoff + pd.DateOffset(months=horizon_months)

    obs_tx = df_money[(df_money['InvoiceDate'] >= obs_start) & (df_money['InvoiceDate'] < cutoff)]
    horizon_tx = df_money[(df_money['InvoiceDate'] >= cutoff) & (df_money['InvoiceDate'] < horizon_end)]

    summary = summary_data_from_transaction_data(
        obs_tx,
        customer_id_col='CustomerID',
        datetime_col='InvoiceDate',
        monetary_value_col='Sales',
        observation_period_end=cutoff,
        freq='D'
    )

    bgf = BetaGeoFitter(penalizer_coef=0.01)
    bgf.fit(summary['frequency'], summary['recency'], summary['T'])

    summary_gg = summary[(summary['frequency'] > 0) & (summary['monetary_value'] > 0)]
    if summary_gg.empty:
        raise RuntimeError('Gamma-Gamma has no positive monetary rows')
    ggf = None
    for pen in [1.0, 0.5, 0.1]:
        try:
            ggf = GammaGammaFitter(penalizer_coef=pen)
            ggf.fit(summary_gg['frequency'], summary_gg['monetary_value'], maxiter=200)
            print(f'Gamma-Gamma converged with penalizer={pen}')
            break
        except Exception as _e:
            ggf = None
    if ggf is None:
        raise RuntimeError('Gamma-Gamma failed to converge')

    # Expected transactions over horizon (days)
    t_days = horizon_months * 30
    exp_txn = bgf.conditional_expected_number_of_purchases_up_to_time(
        t_days, summary['frequency'], summary['recency'], summary['T']
    )

    exp_monetary = ggf.conditional_expected_average_profit(
        summary['frequency'].clip(lower=1), summary['monetary_value'].fillna(summary_gg['monetary_value'].mean())
    )

    pred_monetary = exp_txn * exp_monetary

    actual_monetary = horizon_tx.groupby('CustomerID')['Sales'].sum()
    actual_monetary = actual_monetary.reindex(summary.index).fillna(0.0)

    res_gg_test = evaluate_model(actual_monetary.values, pred_monetary.values, 'BG/NBD+GammaGamma')
    print('BG/NBD + Gamma-Gamma (monetary) test:', res_gg_test)
except Exception as e:
    print(f'Gamma-Gamma skipped: {e}')


Training BG/NBD on 12400 valid customers...


✓ BG/NBD training successful
Val: {'Model': 'BG/NBD', 'MAE': 1.014521343457616, 'RMSE': np.float64(1.9730798084259642)}
Test: {'Model': 'BG/NBD', 'MAE': 1.3897618817566584, 'RMSE': np.float64(3.028801585275431)}


## Technique 4: Sequence Model (Monthly Counts -> MLP proxy)

In [ ]:
# Build monthly count sequences from observation windows
monthly = clean.copy()
monthly['month'] = monthly['InvoiceDate'].dt.to_period('M').dt.to_timestamp()
monthly_counts = monthly.groupby(['CustomerID', 'month']).size().unstack(fill_value=0)
monthly_counts.columns = pd.to_datetime(monthly_counts.columns)

seq_cols = [f'm{i}' for i in range(obs_months, 0, -1)]

def build_seq_features(df_split):
    seq_rows = []
    for _, row in df_split[['CustomerID', 'cutoff_date']].iterrows():
        cid = row['CustomerID']
        cutoff = pd.to_datetime(row['cutoff_date'])
        months = pd.date_range(cutoff - pd.DateOffset(months=obs_months), periods=obs_months, freq='MS')
        if cid in monthly_counts.index:
            counts = monthly_counts.loc[cid].reindex(months, fill_value=0).values
        else:
            counts = [0] * obs_months
        seq_rows.append(counts)
    return pd.DataFrame(seq_rows, columns=seq_cols, index=df_split.index)

X_train_seq = build_seq_features(train_df)
X_val_seq = build_seq_features(val_df)
X_test_seq = build_seq_features(test_df)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor

seq_model = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42))
])

seq_model.fit(X_train_seq, y_train)

pred_seq_val = seq_model.predict(X_val_seq)
pred_seq_test = seq_model.predict(X_test_seq)

res_seq_val = evaluate_model(y_val, pred_seq_val, 'Seq_MLP')
res_seq_test = evaluate_model(y_test, pred_seq_test, 'Seq_MLP')

print('Sequence MLP test:', res_seq_test)


rows = []

# CKPT2 baseline reference
if not baseline_df.empty:
    rows.extend(baseline_df.to_dict('records'))

# Two-Stage
rows.append({'Model': 'TwoStage', 'MAE': res_two_test['MAE'], 'RMSE': res_two_test['RMSE']})

# Count-aware
if 'Poisson_HGB_test' in results_count:
    rows.append({'Model': 'Poisson_HGB', 'MAE': results_count['Poisson_HGB_test']['MAE'], 'RMSE': results_count['Poisson_HGB_test']['RMSE']})
if 'Tweedie_test' in results_count:
    rows.append({'Model': 'Tweedie', 'MAE': results_count['Tweedie_test']['MAE'], 'RMSE': results_count['Tweedie_test']['RMSE']})
if 'XGB_Poisson_test' in results_count:
    rows.append({'Model': 'XGB_Poisson', 'MAE': results_count['XGB_Poisson_test']['MAE'], 'RMSE': results_count['XGB_Poisson_test']['RMSE']})
if 'LGB_Poisson_test' in results_count:
    rows.append({'Model': 'LGB_Poisson', 'MAE': results_count['LGB_Poisson_test']['MAE'], 'RMSE': results_count['LGB_Poisson_test']['RMSE']})
if 'LGB_Tweedie_test' in results_count:
    rows.append({'Model': 'LGB_Tweedie', 'MAE': results_count['LGB_Tweedie_test']['MAE'], 'RMSE': results_count['LGB_Tweedie_test']['RMSE']})

# Sequence model
if 'res_seq_test' in globals():
    rows.append({'Model': 'Seq_MLP', 'MAE': res_seq_test['MAE'], 'RMSE': res_seq_test['RMSE']})

# BG/NBD (count)
if res_bgnbd_test is not None:
    rows.append({'Model': 'BG/NBD', 'MAE': res_bgnbd_test['MAE'], 'RMSE': res_bgnbd_test['RMSE']})

compare_df = pd.DataFrame(rows).sort_values('MAE')
compare_df


In [8]:
rows = []

# CKPT2 baseline reference
if not baseline_df.empty:
    rows.extend(baseline_df.to_dict('records'))

# Two-Stage
rows.append({'Model': 'TwoStage', 'MAE': res_two_test['MAE'], 'RMSE': res_two_test['RMSE']})

# Count-aware
if 'Poisson_HGB_test' in results_count:
    rows.append({'Model': 'Poisson_HGB', 'MAE': results_count['Poisson_HGB_test']['MAE'], 'RMSE': results_count['Poisson_HGB_test']['RMSE']})
if 'Tweedie_test' in results_count:
    rows.append({'Model': 'Tweedie', 'MAE': results_count['Tweedie_test']['MAE'], 'RMSE': results_count['Tweedie_test']['RMSE']})
if 'XGB_Poisson_test' in results_count:
    rows.append({'Model': 'XGB_Poisson', 'MAE': results_count['XGB_Poisson_test']['MAE'], 'RMSE': results_count['XGB_Poisson_test']['RMSE']})
if 'LGB_Poisson_test' in results_count:
    rows.append({'Model': 'LGB_Poisson', 'MAE': results_count['LGB_Poisson_test']['MAE'], 'RMSE': results_count['LGB_Poisson_test']['RMSE']})
if 'LGB_Tweedie_test' in results_count:
    rows.append({'Model': 'LGB_Tweedie', 'MAE': results_count['LGB_Tweedie_test']['MAE'], 'RMSE': results_count['LGB_Tweedie_test']['RMSE']})

# Sequence model
if 'res_seq_test' in globals():
    rows.append({'Model': 'Seq_MLP', 'MAE': res_seq_test['MAE'], 'RMSE': res_seq_test['RMSE']})

# BG/NBD (count)
if res_bgnbd_test is not None:
    rows.append({'Model': 'BG/NBD', 'MAE': res_bgnbd_test['MAE'], 'RMSE': res_bgnbd_test['RMSE']})

compare_df = pd.DataFrame(rows).sort_values('MAE')
compare_df


,Model,MAE,RMSE
0,results_rf_test,1.106082,1.958634
1,results_avg_test,1.110828,1.958989
11,TwoStage,1.111556,1.989355
2,results_mlp_test,1.115126,1.801244
3,results_xgb_test,1.118455,1.850803
4,results_en_test,1.122049,2.160581
12,Poisson_HGB,1.129525,2.271224
5,results_hgb_test,1.130360,2.249780
13,XGB_Poisson,1.138514,2.087571
6,results_et_test,1.180362,2.058104
